# Fermi-Dirac Electronic Occupations

This notebook computes finite-temperature electronic occupations for a tight-binding chain. The physics target is the Fermi-Dirac matrix function

$$f_eta(H-\mu I)=rac{1}{1+e^{eta(H-\mu I)}}.$$

A bounded polynomial approximation acts on the scaled Hamiltonian and is compared with the exact spectral density matrix.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.approximation import chebyshev_fit_function
from qsvt.hamiltonians import tight_binding_chain
from qsvt.polynomials import chebyshev_to_monomial, eval_polynomial
from qsvt.rescaling import rescale_hermitian_to_unit_interval
from qsvt.spectral import (
    apply_function_to_hermitian,
    apply_polynomial_to_hermitian,
    eigh_hermitian,
)

np.set_printoptions(precision=4, suppress=True)

In [ ]:
n_sites = 18
onsite = 0.25 * np.cos(2 * np.pi * np.arange(n_sites) / n_sites)
H = tight_binding_chain(n_sites, hopping=1.0, onsite=onsite, periodic=True)
scaled = rescale_hermitian_to_unit_interval(H)
evals, _ = eigh_hermitian(H)
scaled_evals = np.linalg.eigvalsh(scaled.matrix)

chemical_potential = 0.15
beta = 6.0


def fermi_energy(energy):
    return 1.0 / (1.0 + np.exp(beta * (energy - chemical_potential)))


def fermi_scaled(x):
    energy = scaled.original_from_scaled_value(x)
    return fermi_energy(energy)


cheb_coeffs = chebyshev_fit_function(fermi_scaled, degree=35, num_points=2001)
coeffs = chebyshev_to_monomial(cheb_coeffs)

rho_exact = apply_function_to_hermitian(H, fermi_energy)
rho_poly = apply_polynomial_to_hermitian(scaled.matrix, coeffs)
occupation_error = np.linalg.norm(rho_poly - rho_exact) / np.linalg.norm(rho_exact)
particle_number_exact = np.trace(rho_exact).real
particle_number_poly = np.trace(rho_poly).real
occupation_error, particle_number_exact, particle_number_poly

In [ ]:
fermi_exact_values = fermi_energy(evals)
fermi_poly_values = eval_polynomial(coeffs, scaled_evals)
site_density_exact = np.real(np.diag(rho_exact))
site_density_poly = np.real(np.diag(rho_poly))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)

axes[0].plot(evals, fermi_exact_values, "o", label="exact occupations")
axes[0].plot(evals, fermi_poly_values, "x", label="polynomial")
axes[0].axvline(chemical_potential, color="0.4", linestyle="--")
axes[0].set_xlabel("single-particle energy")
axes[0].set_ylabel("occupation")
axes[0].legend()

axes[1].plot(np.arange(n_sites), site_density_exact, "o-", label="exact")
axes[1].plot(np.arange(n_sites), site_density_poly, "--", label="polynomial")
axes[1].set_xlabel("site")
axes[1].set_ylabel("finite-temperature density")
axes[1].legend()

fig.suptitle("Fermi-Dirac occupation as a spectral matrix function")
plt.show()

In [ ]:
assert occupation_error < 0.02
assert abs(particle_number_poly - particle_number_exact) < 0.05
assert np.all(fermi_poly_values > -0.02)
assert np.all(fermi_poly_values < 1.02)

print(f"relative_density_matrix_error: {occupation_error:.4f}")
print(f"exact_particle_number: {particle_number_exact:.3f}")
print(f"polynomial_particle_number: {particle_number_poly:.3f}")
print("validation: passed")